<a href="https://colab.research.google.com/github/Krishna-matic/python_practice/blob/main/SentimentalAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import re
import string

from sklearn.model_selection import train_test_split

# Text Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [5]:
df = pd.read_csv("/content/drive/MyDrive/DataSetForpython/fake_and_real_news.csv")

In [7]:
df.head()

,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9900 entries, 0 to 9899
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    9900 non-null   object
 1   label   9900 non-null   object
dtypes: object(2)
memory usage: 154.8+ KB


In [10]:
df.shape

(9900, 2)

In [12]:
df.isnull().sum()

,0
Text,0
label,0


In [13]:
df.duplicated().sum()

np.int64(35)

In [15]:
df.drop_duplicates(inplace=True)

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
def clean_text(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove html tags
    text = re.sub(r"<.*?>", "", text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [18]:
df["clean_text"] = df["Text"].apply(clean_text)

In [19]:
df["label"] = df["label"].map({
    "Fake":0,
    "Real":1
})

In [20]:
df.head()

,Text,label,clean_text
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,0,top trump surrogate brutally stabs him in the ...
1,U.S. conservative leader optimistic of common ...,1,us conservative leader optimistic of common gr...
2,"Trump proposes U.S. tax overhaul, stirs concer...",1,trump proposes us tax overhaul stirs concerns ...
3,Court Forces Ohio To Allow Millions Of Illega...,0,court forces ohio to allow millions of illegal...
4,Democrats say Trump agrees to work on immigrat...,1,democrats say trump agrees to work on immigrat...


In [21]:
X = df["clean_text"]

y = df["label"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [23]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_df=0.7
)

In [24]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [25]:
X_test_tfidf = tfidf.transform(X_test)

In [26]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42)

lr.fit(X_train_tfidf, y_train)

LogisticRegression(random_state=42)

In [27]:
y_pred = lr.predict(X_test_tfidf)

In [28]:
predictions = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

predictions.head(10)

,Actual,Predicted
0,1,1
1,1,1
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,1,1
9,1,1


In [30]:

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.988342625443487
Precision: 0.9876796714579056
Recall   : 0.9886947584789312
F1 Score : 0.9881869542886492


In [31]:
def predict_news(news):

    # Clean the input news
    cleaned_news = clean_text(news)

    # Convert text into TF-IDF vector
    news_vector = tfidf.transform([cleaned_news])

    # Predict
    prediction = lr.predict(news_vector)[0]

    # Prediction probability
    probability = lr.predict_proba(news_vector)[0]

    # Display result
    if prediction == 1:
        print("Prediction: REAL NEWS")
        print(f"Confidence: {probability[1]*100:.2f}%")
    else:
        print("Prediction: FAKE NEWS")
        print(f"Confidence: {probability[0]*100:.2f}%")

In [32]:
news = """
NASA successfully launched a new satellite into orbit to study climate change
and improve weather forecasting across the globe.
"""

predict_news(news)

Prediction: FAKE NEWS
Confidence: 57.82%


In [33]:
news = """
Scientists have confirmed that drinking five liters of coffee every day makes
people immortal according to a secret government study.
"""

predict_news(news)

Prediction: FAKE NEWS
Confidence: 57.82%


In [34]:
news = """
The Reserve Bank of India (RBI) announced that it will continue to monitor inflation
and take necessary measures to maintain financial stability. The central bank stated
that its monetary policy decisions will depend on incoming economic data, inflation
trends, and global market conditions. Officials also emphasized the importance of
sustainable economic growth while keeping inflation within the target range.
"""

predict_news(news)

Prediction: REAL NEWS
Confidence: 72.99%


In [35]:
news = """
The Indian Space Research Organisation (ISRO) successfully launched a communication
satellite using the GSLV rocket from the Satish Dhawan Space Centre. Scientists
confirmed that the satellite was placed into its intended orbit and will enhance
communication services across the country.
"""

predict_news(news)

Prediction: FAKE NEWS
Confidence: 60.84%
